In [ ]:
import pandas as pd

In [ ]:
import os
from pathlib import Path
import numpy as np

In [ ]:
N = 1000
P = 100
dataset_folder = f"syn_data_N{N}_P{P}"
results_path = f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/results/'
data_path = os.path.join(results_path,'per_dataset',dataset_folder)
data_path = os.path.expanduser(data_path)

In [ ]:
configs_lst = os.listdir(data_path)

In [ ]:
configs_lst

In [ ]:
dfs_dct_config = {}

for config_nm in configs_lst:
    dfs_lst = []
    for filev in os.listdir(os.path.join(data_path,config_nm)):
        if filev.endswith('.parquet'):
            df = pd.read_parquet(os.path.join(data_path,config_nm,filev))
            dfs_lst.append(df)
    
    if len(dfs_lst)>0:        
        dfs_dct_config[config_nm] = pd.concat(dfs_lst)

In [ ]:
dataset_nan_thresh = 0.9
stat_nan_thresh = 0.9

for kv in dfs_dct_config.keys():
    
    concat_df = dfs_dct_config[kv]
    
    num_datasets = concat_df.shape[0]
    num_stats = concat_df.shape[1]

    print(kv)
    nan_per_dataset = concat_df.isna().sum(axis=1) / num_stats
    
    print('nans per dataset')
    print((nan_per_dataset>0).value_counts())
    print(nan_per_dataset.quantile(dataset_nan_thresh))
    
    nan_per_stat = concat_df.isna().sum(axis=0) / num_datasets

    print('nans per stat')    
    print((nan_per_stat>0).value_counts())
    print(nan_per_stat.quantile(stat_nan_thresh))    

    break

In [ ]:
def drop_nans(concat_df, stat_thresh_q=0.75, dataset_thresh_q=0.25):
    num_datasets = concat_df.shape[0]
    num_stats = concat_df.shape[1]
    
    df = concat_df.copy()

    stat_frac = df.isna().mean(axis=0)
    stat_thresh = stat_frac.quantile(stat_thresh_q)
    # stat_thresh = max(stat_thresh,0.1)
    df = df.loc[:, stat_frac <= stat_thresh]

    ds_frac = df.isna().mean(axis=1)
    dataset_thresh = ds_frac.quantile(dataset_thresh_q)

    df = df.loc[ds_frac <= dataset_thresh]

    dct_outs = {}
    dct_outs["stats kept"] = df.shape[1]
    dct_outs["stats tot"] = num_stats
    dct_outs["datasets kept"] = df.shape[0]
    dct_outs["datasets tot"] = num_datasets
    dct_outs["remaining NaNs"] = df.isna().sum().sum()
    dct_outs["old NaNs"] = concat_df.isna().sum().sum()
    dct_outs["stat_thresh"] = stat_thresh
    dct_outs["dataset_thresh"] = dataset_thresh
    
    print(dct_outs)
    return df,dct_outs

clean = {kv: drop_nans(df) for kv, df in dfs_dct_config.items()}

In [ ]:
clean_dfs = {kv : out_tup[0] for kv,out_tup in clean.items()}
clean_stats = {kv : out_tup[1] for kv,out_tup in clean.items()}

concat_datamatrix = pd.concat(clean_dfs,axis=1,join="inner")

In [ ]:
all_datasets = set.union(*[set(df.index) for kv, df in dfs_dct_config.items()])
excluded_datasets = all_datasets - set(concat_datamatrix.index)
print(excluded_datasets)
print(set([v.split('__')[0] for v in excluded_datasets]))

In [ ]:
import json
from pathlib import Path

print(dataset_folder)
save_folder = os.path.join(os.path.expanduser(results_path),"concat",dataset_folder)
os.makedirs(save_folder,exist_ok=True)

version='st_0.75q_dt_0.25q'

m = concat_datamatrix.copy()
if isinstance(m.columns, pd.MultiIndex):
    m.columns = ["__".join(map(str, c)) for c in m.columns]
    
m.to_parquet(os.path.join(save_folder, f"datamatrix__{len(clean_stats)}configs__{version}.parquet"))

record = {
    "configs": clean_stats,
    "shape": list(m.shape),
    "created": pd.Timestamp.now().isoformat(),
}

with open(os.path.join(save_folder,f"clean_stats__{version}.json"), "w") as f:
    json.dump(record, f, indent=2, default=lambda o : o.item() if hasattr(o, "item") else str(o))

In [ ]:
# clean_stats

In [ ]:
concat_datamatrix

In [ ]:
# drop_nans(dfs_dct_config['pyspoc_internal_stats_fast'],stat_thresh_q=0.75)